<h1>0 - Autograd </h1>

In [ ]:
#Let's make autograd from scratch
#1. First what we want to do is to define a new class of object that could handle mutliplication, addition... and also elementary funcitons
#2. We need also to track the operations and the previous nodes
#3. Compute the local gradient for each node by using a backward chain rule => dL/da = dL/x * dx/da (assuming we know dL/dx)
from numpy import log, exp, tan, cos, sin, tanh, cosh, sinh

class Node():
    def __init__(self, data:float):
        self.data = data
        self.operation = ''
        self.prev = set()
        self.label = ""
        self.grad = 0.0
        self._backward = lambda : None
    
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"
    
    def __add__(self, other):
        if isinstance(other, Node):
            out = self.data + other.data
            out = Node(out)
        else:
            other = Node(other)
            out = self.data + other.data
            out = Node(out)
        out.operation = '+'
        out.prev = {self, other}
        
        def _backward():#L = a + b => dL/da = 1 && dL/db = 1
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward

        return out
    
    def __radd__(self, other):
        return self+other
    
    def __mul__(self, other):
        if isinstance(other, Node):
            out = self.data * other.data
            out = Node(out)
        else:
            other = Node(other)
            out = self.data * other.data
            out = Node(out)
        out.operation = '*'
        out.prev = {self, other}

        def _backward():#L = a * b => dL/da = b && dL/db = a
            self.grad +=  other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out
    
    def __rmul__(self, other):
        return self*other
    
    
    def __truediv__(self, other):
        if isinstance(other, Node):
            out = self.data / other.data
            out = Node(out)
        else:
            other = Node(other)
            out = self.data / other.data
            out = Node(out)
        out.operation = '/'
        out.prev = {self, other}

        def _backward():#L = a/b => dL/da = 1/b && dL/db = -a/(b**2)
            self.grad += 1/other.data * out.grad
            other.grad += -self.data/(other.data**2) * out.grad

        out._backward = _backward
        return out
    
    def __rtruediv__(self, other):
        return Node(other)/self
    
    
    def __sub__(self, other):
        if isinstance(other, Node):
            out = self.data - other.data
            out = Node(out)
        else:
            other = Node(other)
            out = self.data - other.data
            out = Node(out)
        out.operation = '-'
        out.prev = {self, other}

        def _backward():#L = a - b => dL/da = 1 && dL/db = -1
            self.grad += 1.0 * out.grad
            other.grad += -1.0 * out.grad

        out._backward = _backward
        return out
    
    def __rsub__(self, other):
        return Node(other) - self

    def __pow__(self, other):
        if isinstance(other, Node):
            out = self.data**other.data
            out = Node(out)
        else:
            other = Node(other)
            out = self.data**other.data
            out = Node(out)
        out.operation = '**'
        out.prev = {self, other}

        def _backward():#L = a^b => dL/a = b*a^(b-1) && dL/db = ln(a)*a**b
            self.grad += other.data * (self.data ** (other.data - 1)) * out.grad
            other.grad += log(self.data) * (self.data**other.data) * out.grad

        out._backward = _backward
        return out
    
    def __rpow__(self, other):
        return Node(other)**self
    
    def Log(self):
        out = log(self.data)
        out = Node(out)
        out.operation = 'log'
        out.prev = {self}

        def _backward():#L = log(x), dL/dx = 1/x
            self.grad += (1/self.data)*out.grad

        out._backward = _backward  

        return out      

    def Exp(self):
        out = exp(self.data)
        out = Node(out)
        out.operation = 'exp'
        out.prev = {self}

        def _backward():#L = exp(x), dL/dx = exp(x)
            self.grad += exp(self.data)*out.grad

        out._backward = _backward  
        
        return out
    
    def Cos(self):
        out = cos(self.data)
        out = Node(out)
        out.operation = 'cos'
        out.prev = {self}

        def _backward():#L = cos(x), dL/dx = -sin(x)
            self.grad += -sin(self.data)*out.grad

        out._backward = _backward  
        
        return out
    
    def Sin(self):
        out = sin(self.data)
        out = Node(out)
        out.operation = 'sin'
        out.prev = {self}

        def _backward():#L = sin(x), dL/dx = cos(x)
            self.grad += cos(self.data)*out.grad

        out._backward = _backward  
        
        return out
    
    def Tan(self):
        out = tan(self.data)
        out = Node(out)
        out.operation = 'tan'
        out.prev = {self}

        def _backward():#L = tan(x), dL/dx = 1 + tan(x)**2 or dL/dx = 1/cos(x)
            self.grad += (1+tan(self.data)**2)*out.grad

        out._backward = _backward  
        
        return out
    
    def Cosh(self):
        out = cosh(self.data)
        out = Node(out)
        out.operation = 'cosh'
        out.prev = {self}

        def _backward():#L = cosh(x), dL/dx = sinh(x)
            self.grad += sinh(self.data)*out.grad

        out._backward = _backward  
        
        return out
    
    def Sinh(self):
        out = sinh(self.data)
        out = Node(out)
        out.operation = 'sinh'
        out.prev = {self}

        def _backward():#L = sinh(x), dL/dx = cosh(x)
            self.grad += cosh(self.data)*out.grad
        
        out._backward = _backward

        return out
    
    def Tanh(self):
        out = tanh(self.data)
        out = Node(out)
        out.operation = 'tanh'
        out.prev = {self}

        def _backward():#L = tanh(x), dL/dx = 1 - tanh(x)**2
            self.grad += (1 - tanh(self.data)**2)*out.grad
      
        out._backward = _backward

        return out
    
    def __neg__(self):
        return self*(-1)
    
    def backward(self):
        stack = []
        visited = set()
        def build_stack(v):
            if v not in visited:
                visited.add(v)
                for parent in v.prev:
                    build_stack(parent)
                stack.append(v)

        build_stack(self)
        self.grad = 1
        for node in reversed(stack):
            node._backward()
    
    def reset(self):
        stack = []
        visited = set()
        def falling_graph(v):
            if v not in visited:
                visited.add(v)
                for parent in v.prev:
                    falling_graph(parent)
                stack.append(v)
        
        falling_graph(self)

        for node in stack:
            node.grad = 0.0
    

    
x = 1.5 + Node(2.5); x.label = "x"
y = Node(7.2); y.label = "y"
c = x + y; c.label = "c"
L = c**2; L.label = "L"

print(2 - Node(5))

In [ ]:
L.backward()
print(x.grad) #dL/dx(4)
L.reset()

<h1> 0.5 - Let's Try it on a Loss function :  Mean Square Diff </h1>

In [ ]:
#The MSE is defined as follows: 1/n sigma((y^ - (ax + b))**2) the idea is to optimize a and b so as to reduce the error
#Let's Apply autograd in it

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

datax = [1.2, 2.5, 3.1, 4.8, 5.5, 6.0, 7.2, 8.4, 9.1, 9.8]
datay = [7.6, 9.5, 11.6, 14.5, 16.8, 16.7, 19.6, 21.4, 23.7, 24.7]

deriva = 0
derivb = 0

for x,y in zip(datax, datay):
    c = a*x; c.label = 'c'
    d = c + b; d.label = 'd'
    e = y - d; e.label = 'e'
    f = e**2; f.label = 'f'
    f.backward()
    deriva += a.grad
    derivb += b.grad
    f.reset()

deriva /= len(datax)
derivb /= len(datax)


deriva



<h1>I - Optimizer</h1>

<h1> 1 - Gradient Descent </h1>

In [ ]:
#Let's say we have a linear regression problem, and we want to apply gradient descent so as to reduce error
#GD algorithm: xk+1 = xk - gamma*grad(L)

EPOCHS = 1000
LR = 0.01

a = Node(0.0)
b = Node(0.0)

for _ in range(EPOCHS):
    for x,y in zip(datax, datay):
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()
        deriva += a.grad
        derivb += b.grad
        f.reset()

    deriva /= len(datax)
    derivb /= len(datax)

    a.data -= LR*deriva
    b.data -= LR*derivb

print(a.data)

print(b.data)
    

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1> 2 - Stochastic Gradient Descent </h1>

In [ ]:
#Stochastic Gradient Descent is mostly used when the objectif function is a sum of function
#And it basicaly consider that each one function (or the index of each function of the sum) as a random variable
#Following the Uniform pdf

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01

for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()
        a.data -= LR*a.grad
        b.data -= LR*b.grad
        f.reset()


    

print(a.data)

print(b.data)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1> 3 - Momentum  </h1>

In [ ]:
#The Momentum accelerated gradient introduces the Exponential Moving Average, in a sequence Vt = (Beta)Vt-1 + (1-Beta)gradf
#As you will see the algorithm relies on sgd structure

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA = 0.9

vta = a.grad
vtb = b.grad

for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()
        vta = BETA*vta + (1-BETA)*a.grad
        vtb = BETA*vtb + (1-BETA)*b.grad
        a.data -= LR*vta
        b.data -= LR*vtb
        f.reset()


    

print(a.data)

print(b.data)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>3.5 - Nesterov</h1>

In [ ]:
#The Nesterov accelerated relies on the idea of See Before You Leap, in a sequence xk = xk -gamma*beta*vt-1 (See)
#Vt = Beta*Vt-1 + (1-Beta)*gradf(xk)(Before)
#xk+1 = xk - gamma*Vt (You Leap)

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA = 0.9

vta = 0.0
vtb = 0.0

for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        old_a = a.data
        old_b = b.data

        a.data -= LR * BETA * vta
        b.data -= LR * BETA * vtb

        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()
        a.data = old_a
        b.data = old_b

        vta = BETA * vta + (1 - BETA) * a.grad
        vtb = BETA * vtb + (1 - BETA) * b.grad

        a.data -= LR * vta
        b.data -= LR * vtb
        f.reset()


    

print(a.data)

print(b.data)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>4 - RMSProp </h1>

In [ ]:
#The RMSProp (Root Mean Square Propagation) introduces the Exponential Moving Average but on gradf^2, in a sequence Vt = (Beta)Vt-1 + (1-Beta)gradf
#Then we divide by the root of the sequence (we explained why in the theorical part)
import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA = 0.9

vta = 0.0
vtb = 0.0

for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()
        vta = BETA*vta + (1-BETA)*(a.grad**2)
        vtb = BETA*vtb + (1-BETA)*(b.grad**2)
        a.data -= LR*(a.grad/(math.sqrt(vta)))
        b.data -= LR*(b.grad/(math.sqrt(vtb)))
        f.reset()


    

print(a.data)

print(b.data)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>5 - Adam </h1>

In [ ]:
#Here we are, the famous optimizer of all, Adam, which stands for Adaptive Moments, it introduces two sequences, vt and st
#vt is the same as the one for momentum (EMA on gradf) and st which is the same as RMSProp (EMA on gradf^2)
#Don't forget to do Bias Correction to both Vt and St


import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
ITER = 0

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0


for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        ITER += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = BETA1*vta + (1-BETA1)*(a.grad)
        vtb = BETA1*vtb + ((1-BETA1)*(b.grad))
        sta = BETA2*sta + ((1-BETA2)*(a.grad**2))
        stb = BETA2*stb + ((1-BETA2)*(b.grad**2))

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        sta_hat = sta / (1 - BETA2**ITER)
        stb_hat = stb / (1 - BETA2**ITER)
        
        a.data -= LR*(vta_hat/(math.sqrt(sta_hat) + EPS))
        b.data -= LR*(vtb_hat/(math.sqrt(stb_hat) + EPS))
        f.reset()


    

print(a.data)

print(b.data)

#Note that we didn't implement AdamW since it's related more to deeplearning and neural networks since it adds weight decay

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>6 - AdaMax </h1>

In [ ]:
#AdaMax is like a generalized form of Adam, As i have already said in the theorical part
#It was introduced in the same paper as Adam
#It relies on a the infinite norm of Vt => infinite(Vt) = max(gt, B2*vt)


import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
ITER = 0

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0


for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        ITER += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = (BETA1*vta + (1-BETA1)*(a.grad))
        vtb = BETA1*vtb + ((1-BETA1)*(b.grad))
        sta = max(abs(a.grad), BETA2*sta)
        stb = max(abs(b.grad), BETA2*stb)

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        a.data -= LR*(vta_hat/sta)
        b.data -= LR*(vtb_hat/stb)
        f.reset()


    

print(a.data)

print(b.data)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>7 - AMSGrad </h1>

In [ ]:
#AMSGrad solves a problem of convergence of Adam, it avoids deviding by a small number

import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
ITER = 0

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0

sta_hat = 0.0
stb_hat = 0.0


for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        ITER += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = BETA1*vta + (1-BETA1)*(a.grad)
        vtb = BETA1*vtb + ((1-BETA1)*(b.grad))
        sta = BETA2*sta + ((1-BETA2)*(a.grad**2))
        stb = BETA2*stb + ((1-BETA2)*(b.grad**2))

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        sta_hat = max(sta, sta_hat)
        stb_hat = max(stb, stb_hat)
        
        a.data -= LR*(vta_hat/(math.sqrt(sta_hat)+EPS))
        b.data -= LR*(vtb_hat/(math.sqrt(stb_hat)+EPS))
        f.reset()


    

print(a.data)

print(b.data)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>8 - QHAdam</h1>

In [ ]:
#QHAdam stand for Quasi Hyperbolical Adam. Adam relies on Momentum, but recent discoveries proves that momentum doesn't
#prioritize present events which can make the model adapt slowly. So we add a QHM to both moments (vt and st)

import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
ITER = 0
D = 0.5

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0


for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        ITER += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = BETA1*vta + (1-BETA1)*(a.grad)
        vtb = BETA1*vtb + ((1-BETA1)*(b.grad))
        sta = BETA2*sta + ((1-BETA2)*(a.grad**2))
        stb = BETA2*stb + ((1-BETA2)*(b.grad**2))

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        sta_hat = sta / (1 - BETA2**ITER)
        stb_hat = stb / (1 - BETA2**ITER)

        qh_vta = (1-D)*a.grad + D*vta
        qh_vtb = (1-D)*b.grad + D*vtb

        qh_sta = (1-D)*a.grad**2 + D*sta
        qh_stb = (1-D)*b.grad**2 + D*stb
        
        a.data -= LR*(qh_vta/(math.sqrt(qh_sta) + EPS))
        b.data -= LR*(qh_vtb/(math.sqrt(qh_stb) + EPS))
        f.reset()


    

print(a.data)

print(b.data)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>9 - Yogi</h1>

In [ ]:
#Yogi is an optimizer that relies on the difference between vt and gt, it says that the the difference
#could change regarding if the gradient is big or small, so Yogi says we only need the sign of the difference
#not the magnetude

import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 1500
LR = 0.01
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
ITER = 0

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0


for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        ITER += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = BETA1*vta + (1-BETA1)*(a.grad)
        vtb = BETA1*vtb + (1-BETA1)*(b.grad)

        sta = sta - (1 - BETA2) * np.sign(sta - a.grad**2) * (a.grad**2)
        stb = stb - (1 - BETA2) * np.sign(stb - b.grad**2) * (b.grad**2)

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        sta_hat = sta / (1 - BETA2**ITER)
        stb_hat = stb / (1 - BETA2**ITER)
        
        a.data -= LR*(vta_hat/(math.sqrt(sta_hat) + EPS))
        b.data -= LR*(vtb_hat/(math.sqrt(stb_hat) + EPS))
        f.reset()


    

print(a.data)

print(b.data)

#Note that we didn't implement AdamW since it's related more to deeplearning and neural networks since it adds weight decay

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>II - Learning Rate Scheduling</h1>

In [ ]:
# We will apply all the learning rate scheduling methods on Adam

<h1>1 - Constant Decay</h1>

<h2>That's what we all have been using before, fixing a variable LR, we have showed int theory that this methodology isn't effective</h2>

<h1>2 - Step Decay</h1>

In [ ]:
# Reducing the Learning Rate by a factor gamma after every s steps

import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
GAMMA = 0.1
STEPS = 500

iter = 0

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0


for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        iter += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = BETA1*vta + (1-BETA1)*(a.grad)
        vtb = BETA1*vtb + ((1-BETA1)*(b.grad))
        sta = BETA2*sta + ((1-BETA2)*(a.grad**2))
        stb = BETA2*stb + ((1-BETA2)*(b.grad**2))

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        sta_hat = sta / (1 - BETA2**ITER)
        stb_hat = stb / (1 - BETA2**ITER)

        lr = LR*(GAMMA**(math.floor(iter/STEPS)))

        if iter % STEPS == 0:
            print(lr)
        
        a.data -= LR*(vta_hat/(math.sqrt(sta_hat) + EPS))
        b.data -= LR*(vtb_hat/(math.sqrt(stb_hat) + EPS))
        f.reset()


    

print(a.data)

print(b.data)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>3 - Exponential Decay</h1>

In [ ]:
#Exponential Decay is like the continuous version of the step decay, instead of reducing the learning rate
#after s steps, we reduce the lr smoothly and continuously relying on a parameter k

import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
GAMMA = 0.1
STEPS = 500

iter = 0

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0

K = log(2)/STEPS# the formula is actually k = log(n)/s with n is the number that expresses how much we want to reduce an error and STEPS expresses after of many steps we will reduce the error by n, in this case we reduce the error by 2 after X STEPS

for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        iter += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = BETA1*vta + (1-BETA1)*(a.grad)
        vtb = BETA1*vtb + ((1-BETA1)*(b.grad))
        sta = BETA2*sta + ((1-BETA2)*(a.grad**2))
        stb = BETA2*stb + ((1-BETA2)*(b.grad**2))

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        sta_hat = sta / (1 - BETA2**ITER)
        stb_hat = stb / (1 - BETA2**ITER)

        lr = LR*exp(-K*iter)

        if iter % STEPS == 0:
            print(lr)
        
        a.data -= LR*(vta_hat/(math.sqrt(sta_hat) + EPS))
        b.data -= LR*(vtb_hat/(math.sqrt(stb_hat) + EPS))
        f.reset()


    

print(a.data)

print(b.data)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>4 - Cosine Annealing</h1>

In [ ]:
#The idea of the cosine annealing is to follow the cosine curve, especialy the curve the 1+cos(x)/2 function, since it
#avoids the negative part of the cosine function, and it reduces the amplititude by 2 (to keep it = 1)

list = np.linspace(0, 2*np.pi, 1000)

plt.plot(list, (1+np.cos(list))/2)


plt.show

In [ ]:
#Exponential Decay is like the continuous version of the step decay, instead of reducing the learning rate
#after s steps, we reduce the lr smoothly and continuously relying on a parameter k

import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR_min = 1e-7
LR_max = 0.01
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
GAMMA = 0.1
STEPS = 500

iter = 0

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0


for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        iter += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = BETA1*vta + (1-BETA1)*(a.grad)
        vtb = BETA1*vtb + ((1-BETA1)*(b.grad))
        sta = BETA2*sta + ((1-BETA2)*(a.grad**2))
        stb = BETA2*stb + ((1-BETA2)*(b.grad**2))

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        sta_hat = sta / (1 - BETA2**ITER)
        stb_hat = stb / (1 - BETA2**ITER)

        lr = LR_min + (LR_max - LR_min)*(1+np.cos(iter*np.pi/(len(shuffled_datax)*EPOCHS)))/2

        if iter % STEPS == 0:
            print(lr)
        
        a.data -= LR*(vta_hat/(math.sqrt(sta_hat) + EPS))
        b.data -= LR*(vtb_hat/(math.sqrt(stb_hat) + EPS))
        f.reset()


    

print(a.data)

print(b.data)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show

<h1>5 - Warmup</h1>

In [ ]:
#Large models trained with large batch sizes are unstable at the start the first gradients are essentially random, and a high learning rate amplifies that noise into NaNs. 
#Warmup linearly ramps the learning rate from near zero to its peak over the first few percent of training


import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR = 0.01
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
GAMMA = 0.1
STEPS = 500

iter = 0

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0


for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        iter += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = BETA1*vta + (1-BETA1)*(a.grad)
        vtb = BETA1*vtb + ((1-BETA1)*(b.grad))
        sta = BETA2*sta + ((1-BETA2)*(a.grad**2))
        stb = BETA2*stb + ((1-BETA2)*(b.grad**2))

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        sta_hat = sta / (1 - BETA2**ITER)
        stb_hat = stb / (1 - BETA2**ITER)

        lr = LR*(iter/(0.05*len(shuffled_datax)*EPOCHS))#The Twarmup is defined as follows:  Twarmups = n*Steps with n is a percentage of total steps, and obviously steps are the total steps that the algo is gonna pass through, 5% is the typical percentage

        if iter % STEPS == 0:
            print(lr)
        
        a.data -= LR*(vta_hat/(math.sqrt(sta_hat) + EPS))
        b.data -= LR*(vtb_hat/(math.sqrt(stb_hat) + EPS))
        f.reset()


    

print(a.data)

print(b.data)



<h1>5.5 - Typical Matchup: Warmup + Cosine Annealing</h1>

In [ ]:
#This is the most famous combinaison in ml/dl, we use warmup then cosine annealing


import math

a = Node(0.0); a.label = 'a'
b = Node(0.0); b.label = 'b'

indices = np.arange(len(datax))

np.random.shuffle(indices)

shuffled_datax = []
shuffled_datay = []

for i in indices:
    shuffled_datax.append(datax[i])
    shuffled_datay.append(datay[i])

EPOCHS = 500
LR_max = 0.01
LR_min = 1e-7
BETA1 = 0.9
BETA2 = 0.999
EPS= 1e-8
GAMMA = 0.1
STEPS = 500

iter = 0

vta = 0.0
vtb = 0.0

sta = 0.0
stb = 0.0


for _ in range(EPOCHS):
    for x,y in zip(shuffled_datax, shuffled_datay):
        iter += 1
        c = a*x; c.label = 'c'
        d = c + b; d.label = 'd'
        e = y - d; e.label = 'e'
        f = e**2; f.label = 'f'
        f.backward()

        vta = BETA1*vta + (1-BETA1)*(a.grad)
        vtb = BETA1*vtb + ((1-BETA1)*(b.grad))
        sta = BETA2*sta + ((1-BETA2)*(a.grad**2))
        stb = BETA2*stb + ((1-BETA2)*(b.grad**2))

        vta_hat = vta / (1 - BETA1**ITER)
        vtb_hat = vtb / (1 - BETA1**ITER)
        
        sta_hat = sta / (1 - BETA2**ITER)
        stb_hat = stb / (1 - BETA2**ITER)

        if iter < 0.05*len(shuffled_datax)*EPOCHS:
            lr = LR_max*(iter/(0.05*len(shuffled_datax)*EPOCHS))
        else:
            lr = LR_min + (LR_max - LR_min)*(1+np.cos(iter*np.pi/(len(shuffled_datax)*EPOCHS)))/2
        if iter % STEPS == 0:
            print(lr)
        
        a.data -= LR*(vta_hat/(math.sqrt(sta_hat) + EPS))
        b.data -= LR*(vtb_hat/(math.sqrt(stb_hat) + EPS))
        f.reset()


    

print(a.data)

print(b.data)

#not that we can also add warmup to some other methods, in this notebook i treated the famous ones, there is also the cyclical learning rate schedule introduced by Leslie Smith that relies on the triangular function, a pretty interesting method treated in the theory part, it could also be coupled by the exponential decay

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.scatter(datax, datay)
plt.plot(np.array(datax), a.data*np.array(datax) + b.data)

plt.show